# Hydrogen-Bond Analysis for Mechanical Protein Properties

This notebook quantifies hydrogen-bond geometry from the MPPRD/CATH PDB set, relates hydrogen-bond statistics to mechanical properties (`v127` toughness and `v128` strength), and trains an interpretable statistical machine-learning model on hydrogen-bond features.

In [ ]:
from __future__ import annotations

import json
import math
import os
import warnings
from dataclasses import dataclass
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-mprl")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.spatial import cKDTree
from scipy.stats import pearsonr, spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=RuntimeWarning)
sns.set_theme(style="whitegrid", context="talk")

CSV_PATH = Path("/mnt/data1/home/jianquanzhao/data/cath/filtered_All_Mechanical_Vectors_cath_all_fasta_results.csv")
PDB_DIR = Path("/home/jianquanzhao/data/tsinghua/mpprd/pdbs/pdbs")
OUTPUT_DIR = Path("outputs/hbond_analysis")
PLOTS_DIR = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

HBOND_DA_CUTOFF = 3.5
HBOND_HA_CUTOFF = 2.7
HBOND_ANGLE_CUTOFF = 120.0
DONOR_H_DISTANCE_CUTOFF = 1.35
RANDOM_STATE = 7

print(f"CSV: {CSV_PATH}")
print(f"PDB_DIR: {PDB_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR.resolve()}")

: 

In [ ]:
@dataclass(frozen=True)
class AtomRecord:
    index: int
    serial: int
    name: str
    element: str
    resname: str
    chain: str
    resseq: int
    icode: str
    coord: np.ndarray

    @property
    def residue_key(self) -> tuple[str, int, str]:
        return (self.chain, self.resseq, self.icode)


BACKBONE_HEAVY = {"N", "O", "OXT"}
DONOR_ELEMENTS = {"N", "O", "S"}
ACCEPTOR_ELEMENTS = {"O", "N", "S"}
HIS_NAMES = {"HIS", "HID", "HIE", "HIP"}
AA3 = {
    "ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY", "HIS", "HID", "HIE", "HIP",
    "ILE", "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR", "VAL",
}


def _element_from_line(line: str) -> str:
    element = line[76:78].strip().upper()
    if element:
        return element
    name = line[12:16].strip().upper()
    stripped = ''.join(ch for ch in name if ch.isalpha())
    if not stripped:
        return ""
    return stripped[0]


def parse_pdb_atoms(path: Path) -> list[AtomRecord]:
    atoms: list[AtomRecord] = []
    with path.open("r", encoding="utf-8", errors="ignore") as handle:
        for line in handle:
            if not line.startswith(("ATOM", "HETATM")):
                continue
            altloc = line[16].strip()
            if altloc not in {"", "A"}:
                continue
            resname = line[17:20].strip().upper()
            if resname not in AA3:
                continue
            try:
                coord = np.asarray([float(line[30:38]), float(line[38:46]), float(line[46:54])], dtype=np.float64)
                serial = int(line[6:11])
                resseq = int(line[22:26])
            except ValueError:
                continue
            atoms.append(
                AtomRecord(
                    index=len(atoms),
                    serial=serial,
                    name=line[12:16].strip().upper(),
                    element=_element_from_line(line),
                    resname=resname,
                    chain=line[21].strip() or "_",
                    resseq=resseq,
                    icode=line[26].strip(),
                    coord=coord,
                )
            )
    return atoms


def angle_degrees(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
    ba = a - b
    bc = c - b
    denom = np.linalg.norm(ba) * np.linalg.norm(bc)
    if denom <= 1e-12:
        return float("nan")
    cosine = float(np.dot(ba, bc) / denom)
    return float(np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0))))


def atom_role(atom: AtomRecord) -> str:
    return "backbone" if atom.name in BACKBONE_HEAVY else "sidechain"


def acceptor_allowed(atom: AtomRecord, attached_h_count: int) -> bool:
    if atom.element == "O":
        return True
    if atom.element == "S":
        return True
    if atom.element == "N" and atom.resname in HIS_NAMES and atom.name in {"ND1", "NE2"}:
        return attached_h_count == 0
    return False


def sequence_separation(left: AtomRecord, right: AtomRecord) -> int | None:
    if left.chain != right.chain:
        return None
    return abs(int(left.resseq) - int(right.resseq))


def classify_hbond(donor: AtomRecord, acceptor: AtomRecord) -> dict[str, str | int]:
    donor_role = atom_role(donor)
    acceptor_role = atom_role(acceptor)
    sep = sequence_separation(donor, acceptor)
    if sep is None:
        seq_class = "interchain"
    elif sep == 0:
        seq_class = "same_residue"
    elif sep <= 4:
        seq_class = "local"
    else:
        seq_class = "nonlocal"
    return {
        "role_type": f"{donor_role}_to_{acceptor_role}",
        "chem_type": f"{donor.element}_to_{acceptor.element}",
        "seq_class": seq_class,
        "seq_sep": -1 if sep is None else sep,
    }


def detect_hbonds(path: Path) -> tuple[pd.DataFrame, dict[str, float]]:
    atoms = parse_pdb_atoms(path)
    if not atoms:
        return pd.DataFrame(), {"parse_ok": 0.0, "atom_count": 0.0, "residue_count": 0.0}

    coords = np.vstack([atom.coord for atom in atoms])
    heavy_indices = [atom.index for atom in atoms if atom.element != "H"]
    donor_heavy_indices = [atom.index for atom in atoms if atom.element in DONOR_ELEMENTS]
    hydrogen_indices = [atom.index for atom in atoms if atom.element == "H"]

    attached_h_by_heavy = {idx: [] for idx in donor_heavy_indices}
    donor_h_pairs: list[tuple[int, int]] = []
    if donor_heavy_indices and hydrogen_indices:
        donor_tree = cKDTree(coords[donor_heavy_indices])
        for h_idx in hydrogen_indices:
            distance, local_index = donor_tree.query(coords[h_idx], k=1, distance_upper_bound=DONOR_H_DISTANCE_CUTOFF)
            if math.isfinite(float(distance)) and local_index < len(donor_heavy_indices):
                donor_idx = donor_heavy_indices[int(local_index)]
                attached_h_by_heavy[donor_idx].append(h_idx)
                donor_h_pairs.append((donor_idx, h_idx))

    acceptor_indices = [
        atom.index for atom in atoms
        if atom.index in heavy_indices and atom.element in ACCEPTOR_ELEMENTS and acceptor_allowed(atom, len(attached_h_by_heavy.get(atom.index, [])))
    ]
    if not donor_h_pairs or not acceptor_indices:
        residue_count = len({atom.residue_key for atom in atoms if atom.element != "H"})
        return pd.DataFrame(), {"parse_ok": 1.0, "atom_count": float(len(atoms)), "residue_count": float(residue_count)}

    acceptor_tree = cKDTree(coords[acceptor_indices])
    hbonds = []
    seen = set()
    for donor_idx, h_idx in donor_h_pairs:
        donor = atoms[donor_idx]
        hydrogen = atoms[h_idx]
        for local_acc_idx in acceptor_tree.query_ball_point(coords[h_idx], HBOND_HA_CUTOFF):
            acceptor_idx = acceptor_indices[int(local_acc_idx)]
            acceptor = atoms[acceptor_idx]
            if acceptor_idx == donor_idx or acceptor.residue_key == donor.residue_key:
                continue
            da_distance = float(np.linalg.norm(donor.coord - acceptor.coord))
            if da_distance > HBOND_DA_CUTOFF:
                continue
            ha_distance = float(np.linalg.norm(hydrogen.coord - acceptor.coord))
            dha_angle = angle_degrees(donor.coord, hydrogen.coord, acceptor.coord)
            if not math.isfinite(dha_angle) or dha_angle < HBOND_ANGLE_CUTOFF:
                continue
            key = (donor_idx, h_idx, acceptor_idx)
            if key in seen:
                continue
            seen.add(key)
            labels = classify_hbond(donor, acceptor)
            hbonds.append({
                "donor_atom": donor.name,
                "donor_resname": donor.resname,
                "donor_chain": donor.chain,
                "donor_resseq": donor.resseq,
                "hydrogen_atom": hydrogen.name,
                "acceptor_atom": acceptor.name,
                "acceptor_resname": acceptor.resname,
                "acceptor_chain": acceptor.chain,
                "acceptor_resseq": acceptor.resseq,
                "d_a_distance": da_distance,
                "h_a_distance": ha_distance,
                "dha_angle": dha_angle,
                **labels,
            })

    residue_count = len({atom.residue_key for atom in atoms if atom.element != "H"})
    summary = {"parse_ok": 1.0, "atom_count": float(len(atoms)), "residue_count": float(residue_count)}
    return pd.DataFrame(hbonds), summary

In [ ]:
def summarize_hbonds(pdb_id: str, hbonds: pd.DataFrame, parse_summary: dict[str, float], sequence_length: int) -> dict[str, float | str]:
    row: dict[str, float | str] = {
        "PDB_ID": pdb_id,
        "sequence_length": float(sequence_length),
        **parse_summary,
    }
    total = int(len(hbonds))
    row["hbond_count"] = float(total)
    row["hbond_per_residue"] = float(total / max(1, sequence_length))
    if total == 0:
        for name in ["d_a_distance", "h_a_distance", "dha_angle"]:
            for stat in ["mean", "std", "min", "max"]:
                row[f"{name}_{stat}"] = 0.0
        row["strong_hbond_fraction"] = 0.0
        row["weak_hbond_fraction"] = 0.0
    else:
        for name in ["d_a_distance", "h_a_distance", "dha_angle"]:
            values = hbonds[name].astype(float)
            row[f"{name}_mean"] = float(values.mean())
            row[f"{name}_std"] = float(values.std(ddof=0))
            row[f"{name}_min"] = float(values.min())
            row[f"{name}_max"] = float(values.max())
        strong = (hbonds["d_a_distance"] <= 3.0) & (hbonds["dha_angle"] >= 150.0)
        weak = (hbonds["d_a_distance"] > 3.2) | (hbonds["dha_angle"] < 140.0)
        row["strong_hbond_fraction"] = float(strong.mean())
        row["weak_hbond_fraction"] = float(weak.mean())

    expected_role_types = [
        "backbone_to_backbone", "backbone_to_sidechain", "sidechain_to_backbone", "sidechain_to_sidechain"
    ]
    expected_chem_types = ["N_to_O", "O_to_O", "N_to_N", "O_to_N", "S_to_O", "N_to_S", "O_to_S", "S_to_N", "S_to_S"]
    expected_seq_classes = ["local", "nonlocal", "interchain", "same_residue"]
    for column, expected in [("role_type", expected_role_types), ("chem_type", expected_chem_types), ("seq_class", expected_seq_classes)]:
        counts = hbonds[column].value_counts().to_dict() if total else {}
        for key in expected:
            count = float(counts.get(key, 0.0))
            row[f"{column}_{key}_count"] = count
            row[f"{column}_{key}_per_residue"] = count / max(1, sequence_length)
            row[f"{column}_{key}_fraction"] = count / max(1, total)
    return row


features_path = OUTPUT_DIR / "hbond_features.csv"
details_path = OUTPUT_DIR / "hbond_details.parquet"
details_csv_path = OUTPUT_DIR / "hbond_details.csv.gz"

df = pd.read_csv(CSV_PATH, usecols=["PDB_ID", "v127", "v128", "Sequence"])
df["PDB_ID"] = df["PDB_ID"].astype(str)
df["sequence_length"] = df["Sequence"].astype(str).str.len()
df["pdb_path"] = df["PDB_ID"].map(lambda pdb_id: str(PDB_DIR / f"{pdb_id}.pdb"))
df = df[df["pdb_path"].map(lambda value: Path(value).exists())].reset_index(drop=True)
print(f"Matched records with PDB files: {len(df)}")

if features_path.exists():
    feature_df = pd.read_csv(features_path)
    print(f"Loaded cached features: {features_path} shape={feature_df.shape}")
else:
    feature_rows = []
    detail_frames = []
    for i, record in df.iterrows():
        pdb_id = record["PDB_ID"]
        hbonds, parse_summary = detect_hbonds(Path(record["pdb_path"]))
        feature_rows.append(summarize_hbonds(pdb_id, hbonds, parse_summary, int(record["sequence_length"])))
        if not hbonds.empty:
            hbonds = hbonds.copy()
            hbonds.insert(0, "PDB_ID", pdb_id)
            detail_frames.append(hbonds)
        if (i + 1) % 250 == 0:
            print(f"Processed {i + 1}/{len(df)} structures")
    feature_df = pd.DataFrame(feature_rows)
    feature_df.to_csv(features_path, index=False)
    if detail_frames:
        details_df = pd.concat(detail_frames, ignore_index=True)
        try:
            details_df.to_parquet(details_path, index=False)
        except Exception:
            details_df.to_csv(details_csv_path, index=False, compression="gzip")
    print(f"Saved features: {features_path} shape={feature_df.shape}")

analysis_df = df.drop(columns=["sequence_length"]).merge(feature_df, on="PDB_ID", how="inner")
analysis_df["log1p_v127"] = np.log1p(analysis_df["v127"].astype(float))
analysis_df["log1p_v128"] = np.log1p(analysis_df["v128"].astype(float))
analysis_df.to_csv(OUTPUT_DIR / "hbond_analysis_table.csv", index=False)
analysis_df.head()

In [ ]:
target_columns = ["v127", "v128", "log1p_v127", "log1p_v128"]
excluded = {"PDB_ID", "Sequence", "pdb_path", *target_columns}
feature_columns = [col for col in analysis_df.columns if col not in excluded and pd.api.types.is_numeric_dtype(analysis_df[col])]

correlation_rows = []
for feature in feature_columns:
    values = analysis_df[feature].astype(float).replace([np.inf, -np.inf], np.nan)
    if values.nunique(dropna=True) <= 1:
        continue
    valid_feature = values.notna()
    for target in target_columns:
        valid = valid_feature & analysis_df[target].notna()
        if valid.sum() < 4:
            continue
        x = values[valid].to_numpy(dtype=float)
        y = analysis_df.loc[valid, target].to_numpy(dtype=float)
        pearson_value, pearson_p = pearsonr(x, y)
        spearman_value, spearman_p = spearmanr(x, y)
        correlation_rows.append({
            "feature": feature,
            "target": target,
            "pearson": float(pearson_value),
            "pearson_p": float(pearson_p),
            "spearman": float(spearman_value),
            "spearman_p": float(spearman_p),
            "abs_spearman": float(abs(spearman_value)),
        })

corr_df = pd.DataFrame(correlation_rows).sort_values(["target", "abs_spearman"], ascending=[True, False])
corr_df.to_csv(OUTPUT_DIR / "hbond_property_correlations.csv", index=False)

summary_stats = analysis_df[["v127", "v128", "sequence_length", "hbond_count", "hbond_per_residue", "d_a_distance_mean", "h_a_distance_mean", "dha_angle_mean", "strong_hbond_fraction"]].describe().T
summary_stats.to_csv(OUTPUT_DIR / "hbond_summary_statistics.csv")

display(summary_stats)
display(corr_df.groupby("target").head(10))

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(analysis_df["hbond_per_residue"], bins=50, kde=True)
plt.xlabel("Hydrogen bonds per residue")
plt.ylabel("Protein count")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "hbond_per_residue_distribution.png", dpi=200)
plt.close()

for target, label in [("v127", "Toughness v127"), ("v128", "Strength v128")]:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, feature, xlabel in zip(
        axes,
        ["hbond_per_residue", "d_a_distance_mean", "dha_angle_mean"],
        ["Hydrogen bonds per residue", "Mean D-A distance (Å)", "Mean D-H-A angle (degree)"],
    ):
        sns.regplot(data=analysis_df, x=feature, y=target, scatter_kws={"s": 10, "alpha": 0.25}, line_kws={"color": "crimson"}, ax=ax)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(label)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"hbond_geometry_vs_{target}.png", dpi=200)
    plt.close()

top_features = corr_df[corr_df["target"].isin(["log1p_v127", "log1p_v128"])]\
    .sort_values("abs_spearman", ascending=False).head(25)["feature"].unique().tolist()
heatmap_df = corr_df[corr_df["feature"].isin(top_features) & corr_df["target"].isin(["v127", "v128", "log1p_v127", "log1p_v128"])]
heatmap = heatmap_df.pivot(index="feature", columns="target", values="spearman").fillna(0.0)
plt.figure(figsize=(9, max(7, 0.32 * len(heatmap))))
sns.heatmap(heatmap, cmap="vlag", center=0, annot=False)
plt.title("Spearman correlation: hydrogen-bond features vs mechanical properties")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "hbond_property_correlation_heatmap.png", dpi=220)
plt.close()

print(f"Saved plots to {PLOTS_DIR}")

In [ ]:
model_feature_columns = [
    col for col in feature_columns
    if col not in {"parse_ok", "atom_count", "residue_count"} and analysis_df[col].nunique(dropna=True) > 1
]
model_df = analysis_df.dropna(subset=model_feature_columns + ["v127", "v128"]).copy()
X = model_df[model_feature_columns].astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
y_raw = model_df[["v127", "v128"]].astype(float)
y = np.log1p(y_raw.to_numpy(dtype=float))

X_train, X_tmp, y_train, y_tmp, raw_train, raw_tmp = train_test_split(
    X, y, y_raw, test_size=0.2, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test, raw_val, raw_test = train_test_split(
    X_tmp, y_tmp, raw_tmp, test_size=0.5, random_state=RANDOM_STATE
)

model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "rf",
            MultiOutputRegressor(
                RandomForestRegressor(
                    n_estimators=300,
                    max_depth=None,
                    min_samples_leaf=3,
                    max_features="sqrt",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                )
            ),
        ),
    ]
)
model.fit(X_train, y_train)


def evaluate_split(name: str, X_split: pd.DataFrame, raw_true: pd.DataFrame) -> dict[str, float | str]:
    pred = np.expm1(model.predict(X_split))
    true = raw_true[["v127", "v128"]].to_numpy(dtype=float)
    rows = []
    metrics = {"split": name, "n": int(len(X_split))}
    for idx, target_name in enumerate(["toughness_v127", "strength_v128"]):
        metrics[f"{target_name}/r2"] = float(r2_score(true[:, idx], pred[:, idx]))
        metrics[f"{target_name}/mae"] = float(mean_absolute_error(true[:, idx], pred[:, idx]))
        metrics[f"{target_name}/rmse"] = float(np.sqrt(mean_squared_error(true[:, idx], pred[:, idx])))
        metrics[f"{target_name}/spearman"] = float(spearmanr(true[:, idx], pred[:, idx]).correlation)
    return metrics


metrics_df = pd.DataFrame([
    evaluate_split("train", X_train, raw_train),
    evaluate_split("val", X_val, raw_val),
    evaluate_split("test", X_test, raw_test),
])
metrics_df.to_csv(OUTPUT_DIR / "hbond_random_forest_metrics.csv", index=False)
display(metrics_df)

rf = model.named_steps["rf"]
importance = np.mean([est.feature_importances_ for est in rf.estimators_], axis=0)
importance_df = pd.DataFrame({"feature": model_feature_columns, "importance": importance}).sort_values("importance", ascending=False)
importance_df.to_csv(OUTPUT_DIR / "hbond_random_forest_feature_importance.csv", index=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=importance_df.head(20), x="importance", y="feature", color="#4C78A8")
plt.title("Random forest feature importance")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "hbond_random_forest_feature_importance.png", dpi=220)
plt.close()

display(importance_df.head(20))

In [ ]:
top_corr_v127 = corr_df[corr_df["target"] == "log1p_v127"].head(8)
top_corr_v128 = corr_df[corr_df["target"] == "log1p_v128"].head(8)
top_importance = importance_df.head(12)

def markdown_table(frame: pd.DataFrame, float_digits: int = 4) -> str:
    if frame.empty:
        return "No rows."
    formatted = frame.copy()
    for col in formatted.columns:
        if pd.api.types.is_numeric_dtype(formatted[col]):
            formatted[col] = formatted[col].map(lambda value: f"{value:.{float_digits}f}")
    return formatted.to_markdown(index=False)

report = f"""# Hydrogen-Bond Analysis for Mechanical Protein Properties

## Scope

This analysis links hydrogen-bond geometry in the structural set to two mechanical-property labels:

- `v127`: toughness averaged per amino acid.
- `v128`: maximum tensile strength.

Input structures: `{PDB_DIR}`  
Input mechanical-property table: `{CSV_PATH}`

Matched protein structures: **{len(analysis_df)}**.

## Hydrogen-Bond Definition

Because these PDB files contain explicit hydrogen atoms, hydrogen bonds were detected with a geometric donor-hydrogen-acceptor definition:

- donor heavy atom: N/O/S with an attached hydrogen within {DONOR_H_DISTANCE_CUTOFF:.2f} Å;
- acceptor atom: protein O, weak S acceptor, or unprotonated histidine ND1/NE2;
- D-A distance <= {HBOND_DA_CUTOFF:.2f} Å;
- H-A distance <= {HBOND_HA_CUTOFF:.2f} Å;
- D-H-A angle >= {HBOND_ANGLE_CUTOFF:.1f} degrees;
- same-residue donor/acceptor pairs were excluded.

Each hydrogen bond was classified by backbone/sidechain role, donor/acceptor chemistry, and sequence separation (`local`, `nonlocal`, `interchain`).

## Dataset-Level Hydrogen-Bond Statistics

{markdown_table(summary_stats.reset_index().rename(columns={'index': 'feature'}), 4)}

## Correlation With Mechanical Properties

The tables below rank hydrogen-bond features by absolute Spearman correlation against log-transformed targets. Spearman is emphasized because mechanical labels are long-tailed and monotonic structure-property trends are more relevant than strictly linear trends.

### Top Correlations With log1p(v127), Toughness

{markdown_table(top_corr_v127[['feature', 'pearson', 'spearman', 'spearman_p']], 5)}

### Top Correlations With log1p(v128), Strength

{markdown_table(top_corr_v128[['feature', 'pearson', 'spearman', 'spearman_p']], 5)}

Full correlation table: `outputs/hbond_analysis/hbond_property_correlations.csv`.

Main visualizations:

- `outputs/hbond_analysis/plots/hbond_per_residue_distribution.png`
- `outputs/hbond_analysis/plots/hbond_geometry_vs_v127.png`
- `outputs/hbond_analysis/plots/hbond_geometry_vs_v128.png`
- `outputs/hbond_analysis/plots/hbond_property_correlation_heatmap.png`

## Statistical Machine-Learning Model

Chosen method: **random forest regression on hydrogen-bond summary features**.

Rationale:

1. Hydrogen-bond effects are likely nonlinear and threshold-like: one extra nonlocal hydrogen bond may matter differently in a short beta-rich protein than in a long mixed fold.
2. Random forests handle correlated tabular descriptors reasonably well and do not assume a linear relationship.
3. Feature importance gives a first-pass interpretable ranking, useful before moving to deeper geometric or graph models.

The model was trained on `log1p(v127), log1p(v128)` and evaluated after inverse transform to the original mechanical-property scale.

### Metrics

{markdown_table(metrics_df, 4)}

### Top Random-Forest Features

{markdown_table(top_importance, 5)}

Feature importance plot: `outputs/hbond_analysis/plots/hbond_random_forest_feature_importance.png`.

## Interpretation

Hydrogen bonding should be interpreted as a structural network signal rather than only a raw count. The most useful descriptors are expected to combine:

- hydrogen-bond density, normalized by sequence length;
- geometric quality, especially short D-A distance and near-linear D-H-A angle;
- nonlocal or interchain hydrogen bonds, which can resist unfolding pathways more directly than local helix-stabilizing contacts;
- backbone-backbone hydrogen bonds, which often report beta-sheet or regular secondary-structure reinforcement;
- sidechain-mediated hydrogen bonds, which may create sacrificial or load-bearing crosslinks depending on topology.

## Additional Hypotheses

1. **Topology matters more than count alone.** Nonlocal hydrogen bonds and interchain hydrogen bonds should correlate more strongly with strength than local hydrogen bonds, because they couple distant sequence regions and can resist extension.
2. **Geometry quality may separate stiffness from toughness.** Short, linear hydrogen bonds may increase initial resistance, while a larger number of weaker sidechain hydrogen bonds may dissipate energy and improve toughness.
3. **Hydrogen-bond anisotropy matters.** Bonds aligned with the pulling direction should contribute more to tensile strength than bonds orthogonal to the force path. This requires adding pulling-axis or terminal-distance descriptors.
4. **Hydrogen bonds interact with secondary structure.** Beta-rich proteins may gain strength from backbone-backbone hydrogen-bond ladders, while alpha-rich proteins may show weaker direct correlation because helices unzip locally.
5. **Hydrogen bonds and hydrophobic packing are coupled.** A good next model should combine hydrogen-bond network descriptors with solvent-accessible area, contact order, hydrophobic core density, salt bridges, and disulfide/covalent constraints.
6. **Mechanical labels may be dominated by unfolding pathway bottlenecks.** Global aggregate features can miss a small number of critical load-bearing contacts, so graph features around high-contact-order regions may improve prediction.

## Recommended Next Step

Use this hydrogen-bond feature set as an interpretable baseline, then add secondary-structure-aware and contact-order-aware descriptors. If these descriptors improve similarity-split performance, they can be incorporated into the reward model as auxiliary physics-informed features alongside ESM2 embeddings.
"""

(Path("hbond_analysis.md")).write_text(report, encoding="utf-8")
print("Wrote hbond_analysis.md")

metadata = {
    "n_records": int(len(analysis_df)),
    "hbond_criteria": {
        "donor_h_distance_cutoff": DONOR_H_DISTANCE_CUTOFF,
        "d_a_distance_cutoff": HBOND_DA_CUTOFF,
        "h_a_distance_cutoff": HBOND_HA_CUTOFF,
        "dha_angle_cutoff": HBOND_ANGLE_CUTOFF,
    },
    "outputs": {
        "features": str(features_path),
        "analysis_table": str(OUTPUT_DIR / "hbond_analysis_table.csv"),
        "correlations": str(OUTPUT_DIR / "hbond_property_correlations.csv"),
        "model_metrics": str(OUTPUT_DIR / "hbond_random_forest_metrics.csv"),
        "feature_importance": str(OUTPUT_DIR / "hbond_random_forest_feature_importance.csv"),
    },
}
(OUTPUT_DIR / "hbond_analysis_summary.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
metadata